# Séance 1 — Pandas & formats de géodonnées

## TP : explorer, enrichir et exporter un jeu de données spatial

Dans ce TP, on travaille en deux temps :

- **Partie 1 (guidée)** : on révise les bases de `pandas` (documentation, index, exploration, filtrage, tri, agrégation, valeurs manquantes, jointures) sur un petit jeu de 30 villes françaises, avec quelques mini-exercices.
- **Partie 2 (en autonomie)** : on vous donne 4 fichiers de sources différentes (CSV, JSON, GeoJSON, Parquet) décrivant 60 villes françaises — à vous de les charger, les nettoyer, les fusionner, en tirer des indicateurs, et les visualiser sur une carte, en réutilisant les notions vues en cours.

**Durée indicative : 2h30.**


## Installation des dépendances

Si vous n'avez pas encore créé et activé l'environnement virtuel du projet (voir le `README.md` à la racine), exécutez la cellule ci-dessous pour installer les packages nécessaires directement depuis ce notebook. Si votre environnement est déjà prêt, vous pouvez ignorer cette cellule.


In [ ]:
%pip install -q -r ../../requirements.txt


## Rappel : le pipeline data

On vient de voir en cours qu'un pipeline de traitement de données spatiales suit typiquement trois grandes étapes : **collecte** (récupérer la donnée brute — ici, on va la générer nous-mêmes), **validation** (vérifier qu'elle est cohérente : types, valeurs manquantes, valeurs aberrantes) et **transformation** (nettoyer, agréger, croiser avec d'autres sources). Aujourd'hui, on se concentre surtout sur la collecte et un peu de transformation avec `pandas` ; la validation en profondeur arrivera plus tard dans le module (séance 5-6).


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)


## 1. Un premier jeu de données : les villes de France

On part d'une liste de 30 grandes villes françaises avec leur région, leurs coordonnées (latitude/longitude) et leur population. Ce n'est pas un fichier téléchargé : on le construit directement en Python, ce qui nous évite toute dépendance réseau pour cette première partie.


In [ ]:
villes_data = [
    {"ville": "Paris", "region": "Île-de-France", "latitude": 48.8566, "longitude": 2.3522, "population": 2145000},
    {"ville": "Marseille", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.2965, "longitude": 5.3698, "population": 870000},
    {"ville": "Lyon", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7640, "longitude": 4.8357, "population": 522000},
    {"ville": "Toulouse", "region": "Occitanie", "latitude": 43.6047, "longitude": 1.4442, "population": 493000},
    {"ville": "Nice", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.7102, "longitude": 7.2620, "population": 342000},
    {"ville": "Nantes", "region": "Pays de la Loire", "latitude": 47.2184, "longitude": -1.5536, "population": 314000},
    {"ville": "Montpellier", "region": "Occitanie", "latitude": 43.6108, "longitude": 3.8767, "population": 295000},
    {"ville": "Strasbourg", "region": "Grand Est", "latitude": 48.5734, "longitude": 7.7521, "population": 291000},
    {"ville": "Bordeaux", "region": "Nouvelle-Aquitaine", "latitude": 44.8378, "longitude": -0.5792, "population": 260000},
    {"ville": "Lille", "region": "Hauts-de-France", "latitude": 50.6292, "longitude": 3.0573, "population": 233000},
    {"ville": "Rennes", "region": "Bretagne", "latitude": 48.1173, "longitude": -1.6778, "population": 220000},
    {"ville": "Reims", "region": "Grand Est", "latitude": 49.2583, "longitude": 4.0317, "population": 182000},
    {"ville": "Le Havre", "region": "Normandie", "latitude": 49.4944, "longitude": 0.1079, "population": 170000},
    {"ville": "Saint-Étienne", "region": "Auvergne-Rhône-Alpes", "latitude": 45.4397, "longitude": 4.3872, "population": 172000},
    {"ville": "Toulon", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.1242, "longitude": 5.9280, "population": 171000},
    {"ville": "Grenoble", "region": "Auvergne-Rhône-Alpes", "latitude": 45.1885, "longitude": 5.7245, "population": 158000},
    {"ville": "Dijon", "region": "Bourgogne-Franche-Comté", "latitude": 47.3220, "longitude": 5.0415, "population": 156000},
    {"ville": "Angers", "region": "Pays de la Loire", "latitude": 47.4784, "longitude": -0.5632, "population": 152000},
    {"ville": "Nîmes", "region": "Occitanie", "latitude": 43.8367, "longitude": 4.3601, "population": 150000},
    {"ville": "Villeurbanne", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7667, "longitude": 4.8794, "population": 149000},
    {"ville": "Clermont-Ferrand", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7772, "longitude": 3.0870, "population": 147000},
    {"ville": "Le Mans", "region": "Pays de la Loire", "latitude": 48.0061, "longitude": 0.1996, "population": 143000},
    {"ville": "Aix-en-Provence", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.5297, "longitude": 5.4474, "population": 143000},
    {"ville": "Brest", "region": "Bretagne", "latitude": 48.3904, "longitude": -4.4861, "population": 139000},
    {"ville": "Tours", "region": "Centre-Val de Loire", "latitude": 47.3941, "longitude": 0.6848, "population": 136000},
    {"ville": "Amiens", "region": "Hauts-de-France", "latitude": 49.8941, "longitude": 2.2957, "population": 133000},
    {"ville": "Limoges", "region": "Nouvelle-Aquitaine", "latitude": 45.8336, "longitude": 1.2611, "population": 131000},
    {"ville": "Annecy", "region": "Auvergne-Rhône-Alpes", "latitude": 45.8992, "longitude": 6.1294, "population": 128000},
    {"ville": "Perpignan", "region": "Occitanie", "latitude": 42.6887, "longitude": 2.8948, "population": 119000},
    {"ville": "Metz", "region": "Grand Est", "latitude": 49.1193, "longitude": 6.1757, "population": 116000},
]

df_villes = pd.DataFrame(villes_data)
df_villes


### Lire la documentation d'une fonction

Chaque exemple de ce notebook n'illustre qu'un usage parmi beaucoup d'autres. Avant d'utiliser une fonction pandas, prenez le réflexe de consulter sa signature complète : dans Jupyter, `?nom_fonction` ouvre la docstring dans un panneau (interactif uniquement, sans effet dans un export statique du notebook), `Shift+Tab` en survolant un appel de fonction donne un aperçu rapide, et `help(...)` fonctionne partout, y compris en script classique.


In [ ]:
help(pd.DataFrame.sort_values)


### L'index : l'essence du DataFrame

Chaque ligne porte une étiquette — l'index. Par défaut, c'est une suite d'entiers (un `RangeIndex`), mais ce n'est qu'un choix parmi d'autres. L'index est ce qui permet à pandas d'aligner, de fusionner et de sélectionner des données de façon fiable.


In [ ]:
stations = pd.DataFrame({
    "ville": ["Paris", "Lyon", "Marseille"],
    "temperature_c": [12.4, 15.1, 18.7],
})
print(stations.index)

# .loc : sélection par étiquette -- .iloc : sélection par position
print(stations.loc[2, "ville"])
print(stations.iloc[2, 0])

# set_index() donne un nom explicite aux lignes ; reset_index() fait l'inverse
stations_idx = stations.set_index("ville")
print(stations_idx.loc["Marseille"])
stations_idx.reset_index()


### Explorer le DataFrame

Trois réflexes à avoir systématiquement en arrivant sur un nouveau jeu de données : `head()` pour voir un aperçu, `info()` pour les types et les valeurs manquantes, `describe()` pour les statistiques descriptives des colonnes numériques.


In [ ]:
print(df_villes.head())
print()
df_villes.info()


In [ ]:
df_villes.describe()


### Filtrer, trier, grouper

Quelques opérations de base : sélectionner des lignes selon une condition (`df[condition]`), trier avec `sort_values`, et agréger par groupe avec `groupby`.


In [ ]:
# Villes de plus de 200 000 habitants, triées par population décroissante
grandes_villes = df_villes[df_villes["population"] > 200_000].sort_values("population", ascending=False)
grandes_villes


In [ ]:
# Nombre de villes et population totale par région, triés par population totale décroissante
df_villes.groupby("region")["population"].agg(["count", "sum"]).sort_values("sum", ascending=False)


### 🧪 À vous de jouer

1. Filtrez les villes dont la population est **inférieure à 150 000** habitants, puis triez le résultat par latitude **décroissante** (du nord au sud).
2. Regroupez les villes par région : calculez le **nombre de villes** et la **population moyenne** par région, trié par nombre de villes décroissant.


In [ ]:
# TODO 1) Filtrer les villes de moins de 150 000 habitants, trier par latitude décroissante
petites_villes = ...  # à compléter

# TODO 2) Regrouper par région : nombre de villes ("nb_villes") et population moyenne ("population_moyenne"),
#         trié par nb_villes décroissant
stats_region = ...  # à compléter

petites_villes


## 2. Nettoyer et enrichir le jeu de données

En pratique, un jeu de données est rarement aussi propre que celui qu'on vient d'explorer, et l'information utile est souvent répartie sur plusieurs fichiers. On simule quelques valeurs manquantes pour s'entraîner à les repérer et les traiter — un avant-goût de la **validation**, approfondie en séance 5-6 — puis on enrichit nos villes avec un second jeu de données grâce à une **jointure** — on verra les quatre variantes (`inner`, `left`, `right`, `outer`).


In [ ]:
# On simule quelques valeurs manquantes, comme on en rencontre souvent en pratique
df_test = df_villes.copy()
df_test.loc[[3, 17], "population"] = np.nan

print("Valeurs manquantes par colonne :")
print(df_test.isna().sum())

# Deux stratégies courantes : dropna() pour supprimer les lignes, fillna() pour remplacer
mediane_pop = df_test["population"].median()
df_test["population"] = df_test["population"].fillna(mediane_pop)
df_test.loc[[3, 17], ["ville", "population"]]


### `value_counts()` : un piège classique

`value_counts()` compte les occurrences de chaque valeur d'une colonne — mais par défaut, elle **ignore silencieusement** les valeurs manquantes. Sur un petit exemple indépendant :


In [ ]:
exemple = pd.Series([15.1, 18.7, 11.8, None, None])

print("Par défaut :")
print(exemple.value_counts())

print("\nAvec dropna=False :")
print(exemple.value_counts(dropna=False))


À gauche, rien n'indique qu'il manque 2 valeurs : c'est l'erreur la plus fréquente avec cette fonction. Deux autres paramètres utiles : `normalize=True` (proportions plutôt que comptes) et `sort=False` (garde l'ordre d'apparition).


Pour aller plus loin, on croise nos villes avec un second jeu de données : la **superficie** de chaque région (en km²). On utilise `pd.merge` — l'équivalent pandas d'une jointure SQL — pour associer chaque ville à la superficie de sa région, puis on calcule une densité de population approximative.


In [ ]:
superficies_region = pd.DataFrame({
    "region": [
        "Île-de-France", "Provence-Alpes-Côte d'Azur", "Auvergne-Rhône-Alpes",
        "Occitanie", "Pays de la Loire", "Grand Est", "Nouvelle-Aquitaine",
        "Hauts-de-France", "Bretagne", "Normandie", "Bourgogne-Franche-Comté",
        "Centre-Val de Loire",
    ],
    "superficie_km2": [
        12_011, 31_400, 69_711, 72_724, 32_082, 57_433,
        84_061, 31_813, 27_208, 29_907, 47_784, 39_151,
    ],
})

df_villes = df_villes.merge(superficies_region, on="region", how="left")

# Densité illustrative : population de nos 30 villes / superficie de la région
# (pas la population réelle de la région entière, donc à prendre comme un ordre de grandeur)
densite_region = (
    df_villes.groupby(["region", "superficie_km2"])["population"]
    .sum()
    .reset_index()
)
densite_region["densite_hab_km2"] = densite_region["population"] / densite_region["superficie_km2"]
densite_region.sort_values("densite_hab_km2", ascending=False)


### 🧪 À vous de jouer

1. Identifiez la région la plus dense et la région la moins dense (utilisez `idxmax()` / `idxmin()` sur `densite_hab_km2`).
2. Comptez le nombre de villes par région grâce à `value_counts()`, trié par ordre décroissant.


In [ ]:
# TODO 1) Identifier la région la plus dense et la moins dense (idxmax / idxmin sur densite_hab_km2)
region_plus_dense = ...  # à compléter
region_moins_dense = ...  # à compléter

# TODO 2) Compter le nombre de villes par région
nb_villes_region = ...  # à compléter

print(region_plus_dense, "/", region_moins_dense)
nb_villes_region


### Les autres types de jointure : `right` et `outer`

Jusqu'ici on a utilisé `how="left"`. Il existe aussi `how="inner"` (le comportement par défaut de `merge`), `how="right"` (symétrique de `left`) et `how="outer"` (garde tout des deux côtés). La différence ne se voit que lorsque les clés ne correspondent pas parfaitement des deux côtés — illustrons ça avec un petit exemple indépendant de nos villes.


In [ ]:
capteurs = pd.DataFrame({
    "capteur_id": ["C1", "C2", "C3"],
    "ville": ["Paris", "Lyon", "Brest"],
})
mesures_recentes = pd.DataFrame({
    "capteur_id": ["C1", "C2", "C9"],
    "valeur": [21.4, 19.8, 15.0],
})

print("--- how='right' ---")
print(capteurs.merge(mesures_recentes, on="capteur_id", how="right"))

print("\n--- how='outer' ---")
print(capteurs.merge(mesures_recentes, on="capteur_id", how="outer"))


`how="right"` garde tout de `mesures_recentes` : C3 (sans mesure) disparaît, C9 (sans ville connue) reste avec un `NaN`. `how="outer"` garde absolument tout des deux tables : C3 et C9 apparaissent tous les deux, chacun avec un `NaN` là où il manque une correspondance.


### 🧪 À vous de jouer

Calculez la jointure `how="inner"` entre `capteurs` et `mesures_recentes`. Combien de lignes contient-elle, et pourquoi moins que pour `outer` ?


In [ ]:
# TODO : calculer la jointure inner entre capteurs et mesures_recentes
jointure_inner = ...  # à compléter

jointure_inner


## Partie 2 — En autonomie : les villes de France, version complète

Cette fois, on ne vous donne plus le code : seulement des **consignes** et des **cellules vides** à remplir. Vous êtes libres dans votre façon de faire, du moment que vous arrivez au résultat demandé — n'hésitez pas à consulter `help(...)`, les slides du cours, ou la documentation pandas / geopandas / plotly / folium.

On vous fournit 4 fichiers, chacun produit par un service différent, dans `data/` :

| Fichier | Format | Contenu |
|---|---|---|
| `villes_geo.geojson` | GeoJSON | `ville`, `region`, géométrie (point) |
| `villes_demographie.csv` | CSV | `ville`, `population`, `age_moyen` |
| `villes_economie.json` | JSON | `ville`, `pib_par_habitant` |
| `villes_climat.parquet` | Parquet | `ville`, `temperature_moyenne_c` |

Comme souvent en pratique, ces fichiers contiennent chacun quelques défauts (valeurs manquantes, voire pire) qu'il faudra repérer et traiter avant de pouvoir les exploiter sereinement.

### A. Charger les 4 fichiers

Chargez chacun des 4 fichiers avec la fonction pandas / geopandas adaptée à son format, et affichez un aperçu (`head()`) de chacun.


### B. Explorer et nettoyer chaque source

Pour chacun des 4 jeux de données :

1. Comptez les valeurs manquantes par colonne.
2. Choisissez une stratégie de traitement adaptée à chaque colonne (suppression ou remplacement) et appliquez-la.

Vérifiez aussi les valeurs **uniques** de la colonne `region` du fichier géographique : un problème de saisie s'y est glissé (deux libellés différents désignent en réalité la même région). Repérez-le et corrigez-le.


*✏️ Notez ici vos choix de nettoyage (colonne par colonne) et pourquoi.*


### C. Fusionner les 4 sources

Combinez les 4 jeux de données nettoyés en une seule table, sur la clé commune `ville`. Réfléchissez au type de jointure (`inner` / `left` / `right` / `outer`) le plus adapté ici, et vérifiez qu'aucune ville n'a été perdue ou dupliquée de façon inattendue (comparez le nombre de lignes avant/après).

💡 Si vous avez supprimé des lignes pour nettoyer une des sources, une jointure `left` sur le référentiel géographique peut faire réapparaître des `NaN` pour ces villes-là : c'est normal, et c'est un bon réflexe de le vérifier.


### D. Découper la France en 4 quadrants

À partir de la géométrie, retrouvez la latitude et la longitude de chaque ville (`geometry.y` / `geometry.x`). Calculez la latitude et la longitude **médianes** de l'ensemble des villes, puis créez une colonne `quadrant` (`"Nord-Ouest"`, `"Nord-Est"`, `"Sud-Ouest"`, `"Sud-Est"`) à l'aide d'une fonction personnalisée appliquée ligne à ligne (cf. slides : « Filtrer / créer une colonne avec une fonction personnalisée »).


### E. Calculer des indicateurs agrégés

Avant de conclure quoi que ce soit, regardez les statistiques descriptives (`describe()`) de vos indicateurs : une valeur de température vous semble-t-elle physiquement impossible pour la France ? Traitez-la comme vous traiteriez une valeur manquante.

Ensuite, pour chaque **quadrant**, puis pour chaque **région**, calculez en une seule fois (`agg`) : population totale, PIB par habitant moyen, âge moyen, température moyenne, et nombre de villes. Triez les résultats par population totale décroissante.


*✏️ Qu'en déduisez-vous ? (2-3 phrases sur les écarts entre quadrants / régions : population, richesse, climat...)*


### F. Exploration rapide avec `pandas.plot`

Produisez 2 ou 3 graphiques rapides avec `.plot()` pour explorer vos indicateurs (par exemple : population par quadrant, PIB par habitant vs âge moyen, distribution de la température). L'objectif ici est la rapidité, pas l'esthétique.


### G. Carte interactive avec Plotly

Créez une carte Plotly (`scatter_geo` ou équivalent) de vos 60 villes : la couleur représente le quadrant (ou une région), la taille un indicateur de votre choix (population, PIB...), et l'info-bulle (`hover_name` / `hover_data`) donne le détail par ville. Exportez-la en HTML (`fig.write_html(...)`).


### H. Carte interactive avec Folium

Créez une carte Folium : un marqueur par ville, coloré selon le quadrant (ou la région), avec une info dans le popup (nom, population, indicateur de votre choix). Exportez-la en HTML (`carte.save(...)`).


### I. Conclusion

En 3 à 5 phrases : que retenez-vous de la comparaison entre quadrants / régions ? Et pour quel usage choisiriez-vous `pandas.plot`, Plotly ou Folium (exploration personnelle, rapport, diffusion web) ?

*✏️ Répondez directement dans cette cellule.*


## Ce qu'on a vu aujourd'hui

- Les bases de `pandas` pour explorer un jeu de données tabulaire : `head`, `info`, `describe`, filtrage, tri, `groupby`.
- Comment lire la documentation d'une fonction (`help`, `?`, Shift+Tab) et pourquoi l'index (`.loc`/`.iloc`, `set_index`/`reset_index`) est au cœur du DataFrame.
- Comment repérer et traiter des valeurs manquantes (`isna`, `dropna`, `fillna`), des valeurs aberrantes, et des doublons de catégorie dus à des erreurs de saisie.
- Comment enrichir un jeu de données par jointure avec `pd.merge` (`inner`/`left`/`right`/`outer`), l'équivalent pandas des `JOIN` SQL — y compris en fusionnant 4 sources de formats différents (CSV, JSON, GeoJSON, Parquet).
- Comment créer une colonne dérivée avec une fonction personnalisée (ici, un découpage géographique en quadrants).
- Comment calculer plusieurs agrégations à la fois avec `agg`, pour comparer des indicateurs entre groupes.
- Comment visualiser rapidement des données avec `pandas.plot()`, puis produire des cartes interactives diffusables avec `Plotly` et `folium` (export HTML).

**La semaine prochaine (séance 2)** : on quitte les fichiers locaux pour aller chercher de la donnée via des APIs — on construira notre propre micro-API avant d'interroger une vraie API publique (USGS, séismes).
